In [77]:
import torch
import torch.nn as nn
import urllib.request
import os

In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [79]:
class GramMatrix(nn.Module):
    def forward(self, x):
        b, c, h, w = x.size()
        features = x.view(b * c, h * w)
        G = torch.mm(features, features.t())
        return G.div(b * c * h * w)

In [80]:
class TotalVariationLoss(nn.Module):
    def forward(self, x):
        h_variance = torch.mean(torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]))
        w_variance = torch.mean(torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]))
        return h_variance + w_variance

In [81]:
class GeospatialStyleTransfer(nn.Module):
    def __init__(self):
        super(GeospatialStyleTransfer, self).__init__()
        
        self.conv1_1 = nn.Conv2d(1,64 , kernel_size=3 , padding= 1)
        self.relu1_1 = nn.ReLU(inplace=True)
        self.conv1_2 = nn.Conv2d(64,64, kernel_size=3 , padding=1)
        self.relu1_2 = nn.ReLU(inplace=True)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu2_1 = nn.ReLU(inplace=True)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.relu2_2 = nn.ReLU(inplace=True)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding= 1)
        self.relu3_1 = nn.ReLU(inplace=True)
        self.conv3_2 = nn.Conv2d(256,256, kernel_size=3, padding= 1)
        self.relu3_2 = nn.ReLU(inplace=True)
        self.conv3_3 = nn.Conv2d(256,256, kernel_size=3, padding= 1)
        self.relu3_3 = nn.ReLU(inplace=True)
        self.conv3_4 = nn.Conv2d(256,256, kernel_size=3, padding= 1)
        self.relu3_4 = nn.ReLU(inplace=True)
        self.pool3 = nn.AvgPool2d(kernel_size=2, stride= 2)
        
        self.conv4_1 = nn.Conv2d(256,512,kernel_size=3, padding= 1)
        self.relu4_1 = nn.ReLU(inplace=True)
        self.conv4_2 = nn.Conv2d(512,512,kernel_size=3, padding=1)
        self.relu4_2 = nn.ReLU(inplace=True)
        self.conv4_3 = nn.Conv2d(512,512,kernel_size=3, padding=1)
        self.relu4_3 = nn.ReLU(inplace=True)
    
    def forward(self, x):
        out = {}
        out['conv1_1'] = self.relu1_1(self.conv1_1(x))
        x = self.pool1(self.relu1_2(self.conv1_2(out["conv1_1"])))
        
        out['conv2_1'] = self.relu2_1(self.conv2_1(x))
        x = self.pool2(self.relu2_2(self.conv2_2(out["conv2_1"])))
        
        out['conv3_1'] = self.relu3_1(self.conv3_1(x))
        x = self.relu3_2(self.conv3_2(out['conv3_1']))
        x = self.relu3_3(self.conv3_3(x))
        x = self.pool3(self.relu3_4(self.conv3_4(x))) 
        
        out['conv4_1'] = self.relu4_1(self.conv4_1(x))
        out['conv4_2'] = self.relu4_2(self.conv4_2(out['conv4_1']))
        out['conv4_3'] = self.relu4_3(self.conv4_3(out["conv4_2"]))
        
        style_maps = [out['conv3_1'], out['conv4_1'], out['conv4_2']]
        content_maps = [out['conv4_3']]
        return content_maps, style_maps

In [ ]:
def load_and_inject_weights(model):
    weight_url = 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth'
    weight_path = 'vgg19_raw.pth'
    
    if not os.path.exists(weight_path):
        print("Downloading raw state_dict VGG19...")
        urllib.request.urlretrieve(weight_url, weight_path)
    
    raw_weights = torch.load(weight_path)
    
    print("Starting injection...")
    model_dict = model.state_dict()
    
    layer_mapping = {
        'conv1_1': 'features.0', 'conv1_2': 'features.2',
        'conv2_1': 'features.5', 'conv2_2': 'features.7',
        'conv3_1': 'features.10', 'conv3_2': 'features.12', 
        'conv3_3': 'features.14', 'conv3_4': 'features.16',
        'conv4_1': 'features.19', 'conv4_2': 'features.21',
        'conv4_3': 'features.23',
    }
    
    for custom_name, raw_name in layer_mapping.items():
        weight_key = f"{raw_name}.weight"
        bias_key = f"{raw_name}.bias"
        
        if custom_name == 'conv1_1':
            rgb_weight = raw_weights[weight_key]
            gray_weight = rgb_weight.sum(dim=1, keepdim=True) 
            model_dict[f"{custom_name}.weight"].copy_(gray_weight)
        else:
            model_dict[f"{custom_name}.weight"].copy_(raw_weights[weight_key])
            
        model_dict[f"{custom_name}.bias"].copy_(raw_weights[bias_key])
        
    for param in model.parameters():
        param.requires_grad = False
        
    print("Weight injection successful. Architecture is ready to use.")
    return model

L-BFGS

In [83]:
import torch.optim as optim
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image

In [84]:
def calculation_mse_loss(generated, target):
    return torch.mean((generated - target) ** 2)

In [ ]:
def train_hybrid_heightmap(content_path, style_path,output_path):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((512, 512)),
        transforms.ToTensor()
    ])
    def load_img(img_path):
        img = Image.open(img_path)
        return transform(img).unsqueeze(0).to(device)
   
    print("Load data...")
    content_img = load_img(content_path)
    style_img = load_img(style_path)
    
    generated_img = nn.Parameter(content_img.clone())
    
    model = GeospatialStyleTransfer().to(device).eval()
    model = load_and_inject_weights(model)
    
    gram_calc = GramMatrix().to(device)
    tv_loss_calc = TotalVariationLoss().to(device)
    
    print("Extracting features from image...")
    with torch.no_grad():
        norm_content = (content_img - 0.449) / 0.226
        norm_style = (style_img - 0.449) / 0.226
        
        target_content_features, _ = model(norm_content)
        _, target_style_features = model(norm_style)
        
        target_gram_matrices = [gram_calc(sf) for sf in target_style_features]
        
    optimizer = optim.LBFGS([generated_img], max_iter = 500, lr=1.0)
    
    alpha = 1
    beta = 1e9
    gamma = 1e-4
    
    print("Starting the L-BFGS Optimization Process...")
    run= [0]
    
    def closure():
        optimizer.zero_grad()
        generated_img.data.clamp_(0,1)
        norm_gen = (generated_img - 0.449) / 0.226
        gen_content_features, gen_style_features = model(norm_gen)
        content_loss = calculation_mse_loss(gen_content_features[0], target_content_features[0])
        
        style_loss = 0
        for gen_sf, target_gm in zip(gen_style_features, target_gram_matrices):
            gen_gm = gram_calc(gen_sf)
            style_loss += calculation_mse_loss(gen_gm, target_gm)
        
        tv_loss = tv_loss_calc(generated_img)
        total_loss = (alpha*content_loss) + (beta*style_loss) + (gamma*tv_loss)
        total_loss.backward()
        
        run[0] += 1
        if run[0] % 50 == 0:
            print(f"Iteration: {run[0]:03d}/500 | Total Loss: {total_loss.item():.4f}")
        return total_loss
        
    optimizer.step(closure)
    
    generated_img.data.clamp_(0,1)
    save_image(generated_img, output_path)
    print(f"\nDone! Hybrid Heightmap has been successfully saved in: {output_path}")
        
        

In [86]:
if __name__ == "__main__":
    CONTENT = "img\contents\PerlinMap_1.png" 
    STYLE = "img/style/Style_1.png"
    OUTPUT = "img/output/hybrid_terrain.png"
    
    train_hybrid_heightmap(CONTENT, STYLE, OUTPUT)

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Isimo\AppData\Local\Temp\ipykernel_16596\2282893471.py:2: SyntaxWarning: invalid escape sequence '\c'
  CONTENT = "img\contents\PerlinMap_1.png"


Load data...
Starting injection...
Weight injection successful. Architecture is ready to use.
Extracting features from image...
Starting the L-BFGS Optimization Process...
Iteration: 050/500 | Total Loss: 0.1281
Iteration: 100/500 | Total Loss: 0.1248
Iteration: 150/500 | Total Loss: 0.1243
Iteration: 200/500 | Total Loss: 0.1240
Iteration: 250/500 | Total Loss: 0.1239
Iteration: 300/500 | Total Loss: 0.1239
Iteration: 350/500 | Total Loss: 0.1238
Iteration: 400/500 | Total Loss: 0.1237
Iteration: 450/500 | Total Loss: 0.1237
Iteration: 500/500 | Total Loss: 0.1237

Done! Hybrid Heightmap has been successfully saved in: img/output/hybrid_terrain.png
